# Nord Pool Day-Ahead Prices

This notebook demonstrates how to retrieve day-ahead electricity prices from the Nord Pool Data Portal using `nexa-marketdata`.

## Prerequisites

You need a Nord Pool account with API access. Set your credentials as environment variables:

```bash
export NORDPOOL_USERNAME="your-username"
export NORDPOOL_PASSWORD="your-password"
```

Or pass them directly when creating the client (shown below).

In [ ]:
import datetime

from nexa_marketdata import NexaClient
from nexa_marketdata.types import BiddingZone, Resolution

## Create the client

The client picks up credentials from the `NORDPOOL_USERNAME` and `NORDPOOL_PASSWORD` environment variables automatically. You can also pass them explicitly.

In [ ]:
# Credentials from environment variables (recommended)
client = NexaClient()

# Or pass credentials explicitly:
# client = NexaClient(nordpool_username="user@example.com", nordpool_password="secret")

## Fetch hourly day-ahead prices

Retrieve hourly day-ahead prices for a single bidding zone over a date range.
The result is a `pandas.DataFrame` with a UTC-aware `DatetimeIndex` and a `price_eur_mwh` column containing `Decimal` values.

In [ ]:
prices = client.day_ahead_prices(
    zone=BiddingZone.NO2,
    start=datetime.date(2025, 1, 1),
    end=datetime.date(2025, 1, 7),
)

print(f"Rows: {len(prices)}")
print(f"Index timezone: {prices.index.tz}")
print(f"Dtype: {prices['price_eur_mwh'].dtype}")
prices.head()

## Supported bidding zones

Nord Pool covers the following zones:

In [ ]:
nordpool_zones = [
    BiddingZone.NO1, BiddingZone.NO2, BiddingZone.NO3, BiddingZone.NO4, BiddingZone.NO5,
    BiddingZone.SE1, BiddingZone.SE2, BiddingZone.SE3, BiddingZone.SE4,
    BiddingZone.DK1, BiddingZone.DK2,
    BiddingZone.FI,
    BiddingZone.DE_LU, BiddingZone.AT, BiddingZone.BE, BiddingZone.NL,
    BiddingZone.FR, BiddingZone.PL,
]
print("Supported zones:", ", ".join(z.value for z in nordpool_zones))

## Compare prices across Nordic zones

Fetch prices for all five Norwegian zones on the same day and compare them.

In [ ]:
import pandas as pd

date = datetime.date(2025, 1, 6)
norwegian_zones = [BiddingZone.NO1, BiddingZone.NO2, BiddingZone.NO3, BiddingZone.NO4, BiddingZone.NO5]

frames = {}
for zone in norwegian_zones:
    df = client.day_ahead_prices(zone=zone, start=date, end=date)
    frames[zone.value] = df["price_eur_mwh"].apply(float)  # convert for display only

comparison = pd.DataFrame(frames)
comparison.index = comparison.index.strftime("%H:%M")
comparison.head()

In [ ]:
comparison.plot(
    title=f"NO Day-Ahead Prices — {date} (EUR/MWh)",
    ylabel="EUR/MWh",
    xlabel="Hour (UTC)",
    figsize=(12, 5),
)

## 15-minute resolution (MTU)

Following the EU MTU transition on 30 September 2025, Nord Pool publishes 15-minute prices.
Request them with `Resolution.MINUTES_15`.

In [ ]:
prices_15min = client.day_ahead_prices(
    zone=BiddingZone.NO2,
    start=datetime.date(2025, 10, 1),
    end=datetime.date(2025, 10, 1),
    resolution=Resolution.MINUTES_15,
)

print(f"Rows (expect 96 for a full day): {len(prices_15min)}")
prices_15min.head()

## Summary statistics

In [ ]:
week = client.day_ahead_prices(
    zone=BiddingZone.NO2,
    start=datetime.date(2025, 1, 1),
    end=datetime.date(2025, 1, 7),
)

float_prices = week["price_eur_mwh"].apply(float)
print(f"Min:    {float_prices.min():.2f} EUR/MWh")
print(f"Max:    {float_prices.max():.2f} EUR/MWh")
print(f"Mean:   {float_prices.mean():.2f} EUR/MWh")
print(f"Median: {float_prices.median():.2f} EUR/MWh")